In [ ]:
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import portfolio_optimizer.market_modelling.black_scholes as bs
import portfolio_optimizer.market_modelling.dsvi as dsvi
import portfolio_optimizer.market_modelling.svcj as svcj

from scipy.optimize import minimize
from portfolio_optimizer.portfolio_models.short_spx_bond_overlay import ShortSPXPutStrategy


In [ ]:
# --- Real options chain data from Aug 15th 2026
spx_chain_data = """
6900	.2283
7200	.1907
7375	.1698
7475	.1586
7550	.1506
7625	.1433
7700	.1367
7750	.1328
7825	.1280
7875	.1256
7925	.1238
8000	.1213
8050	.1210
8100	.1214
8300	.1330
"""
spx_chain_dtes = 44.0
spot_spx = 7786.00
spot_vix = 0.1425
strikes = []
ivs = []

for row in spx_chain_data.strip().split('\n'):
    strike, iv = row.split('\t')
    strikes.append(float(strike))
    ivs.append(float(iv))

svi = dsvi.DynamicSVI(np.array(strikes), np.array(ivs), spot_spx, spx_chain_dtes / 365)
market_sim = svcj.SVCJSimulation()
spx, vix = market_sim.generate_path(spot_spx, spot_vix)
vix3m = market_sim.derive_vix3m(vix)
trade_strategy = ShortSPXPutStrategy(spx, vix, vix3m, svi)
trade_strategy.run_simulation(nav=8000000, monthly_distribution=12500, notional_leverage=0.5)